# Speculative Decoding Engine — Cloud GPU Benchmark (Colab / Kaggle)

Turnkey notebook for executing full Speculative Decoding benchmarks on GPU (T4 / A100 / L4 / V100).

### Workflow:
1. **Setup:** Environment inspection, repository cloning, dependency installation.
2. **Verify:** Fast in-memory unit tests (`pytest tests/ -v`).
3. **Benchmark:** End-to-end evaluation comparing Baseline vs. Speculative decoding across $\gamma \in \{2, 4, 6, 8\}$.
4. **Charts & Export:** Render publication-grade figures, format copy-pasteable Markdown table for `README.md`, and download `gpu_artifacts.zip`.

## Cell 1: Environment Setup

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

import os
import subprocess
import sys
import torch

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: No CUDA GPU detected. Running on CPU will be slow.")

# 2. Clone repository if src/ is not present in the current working directory
if not os.path.isdir("src"):
    repo_name = "Speculative-Decoding-Engine"
    repo_url = "https://github.com/Medmad710/Speculative-Decoding-Engine.git"
    if not os.path.isdir(repo_name):
        print(f"Cloning {repo_url}...")
        subprocess.run(["git", "clone", repo_url], check=True)
    os.chdir(repo_name)
    print(f"Changed working directory to {os.getcwd()}")
else:
    print(f"Using existing repository in {os.getcwd()}")

# 3. Install requirements
!pip install -q -r requirements.txt
print("Dependencies successfully installed.")

## Cell 2: Verify Correctness via Test Suite

In [ ]:
# Run test suite to verify greedy equivalence, EOS handling, and DynamicCache integrity
!pytest tests/ -v

## Cell 3: Run Full Benchmark (Baseline vs. Speculative $\gamma \in \{2, 4, 6, 8\}$)

In [ ]:
# Hardware detector automatically selects model pair:
# - CUDA >= 12GB: Target Qwen3-4B / Draft Qwen3-0.6B (bfloat16)
# - CUDA < 12GB:  Target Qwen2.5-1.5B / Draft Qwen2.5-0.5B (float16)
!python benchmarks/run_benchmark.py \
    --max-new-tokens 256 \
    --gamma 2 4 6 8 \
    --warmup 2

## Cell 4: Generate Publication Charts & Export Artifacts

In [ ]:
import json
import os
import zipfile
import torch
from IPython.display import Image, display

# 1. Generate publication-quality PNG charts
!python benchmarks/plot_results.py

# 2. Display figures inline
print("\n=== SPEEDUP VS DRAFT WINDOW SIZE (GAMMA) ===")
display(Image("assets/speedup_vs_gamma.png"))

print("\n=== ACCEPTANCE RATE BY DOMAIN ===")
display(Image("assets/acceptance_by_domain.png"))

# 3. Format and print copy-pasteable Markdown table for README.md
with open("benchmarks/results.json") as f:
    data = json.load(f)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
meta = data.get("meta", {})
target_m = meta.get("target_model", "Target")
draft_m = meta.get("draft_model", "Draft")
max_tokens = meta.get("max_new_tokens", 256)

baseline_results = data.get("baseline", [])
spec_results = data.get("speculative", [])

b_tps = (sum(r["throughput_tps"] for r in baseline_results) / len(baseline_results)) if baseline_results else 0.0
b_ttft = (sum(r["ttft_ms"] for r in baseline_results) / len(baseline_results)) if baseline_results else 0.0
b_vram = max((r.get("peak_vram_mb", 0.0) for r in baseline_results), default=0.0)

print("\n" + "=" * 75)
print("COPY-PASTEABLE KEY RESULTS TABLE FOR README.md:")
print("=" * 75)
print(f"## Key Results\n")
print(f"> Results on {gpu_name} · {target_m} (target) + {draft_m} (draft) · {max_tokens} new tokens · greedy decoding\n")
print(f"| Decoder | γ | Throughput (tok/s) | Speedup | Mean α | TTFT (ms) | Peak VRAM |")
print(f"|---------|---|-------------------|---------|--------|-----------|-----------|")
print(f"| Baseline | — | {b_tps:.1f} | 1.00× | — | {b_ttft:.1f} | {b_vram:.0f} MB |")

for g in meta.get("gamma_values", [2, 4, 6, 8]):
    g_res = [r for r in spec_results if r.get("gamma") == g]
    if g_res:
        tps = sum(r["throughput_tps"] for r in g_res) / len(g_res)
        speedup = sum(r["speedup_ratio"] for r in g_res) / len(g_res)
        alphas = [r["mean_acceptance_rate"] for r in g_res if r.get("mean_acceptance_rate") is not None]
        m_alpha = sum(alphas) / len(alphas) if alphas else 0.0
        ttft = sum(r["ttft_ms"] for r in g_res) / len(g_res)
        vram = max((r.get("peak_vram_mb", 0.0) for r in g_res), default=0.0)
        print(f"| Speculative | {g} | {tps:.1f} | {speedup:.2f}× | {m_alpha:.3f} | {ttft:.1f} | {vram:.0f} MB |")
print("=" * 75 + "\n")

# 4. Bundle artifacts into a zip archive
zip_filename = "gpu_artifacts.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in ["assets/speedup_vs_gamma.png", "assets/acceptance_by_domain.png", "benchmarks/results.json"]:
        if os.path.exists(file_path):
            zipf.write(file_path)
print(f"[Export] Artifacts bundled into '{zip_filename}'")

# 5. Trigger download if running in Google Colab
try:
    from google.colab import files
    files.download(zip_filename)
    print("[Colab] Download triggered for gpu_artifacts.zip")
except ImportError:
    print(f"[Local/Kaggle] Download archive directly from: {os.path.abspath(zip_filename)}")